## Custom Middleware

> https://docs.langchain.com/oss/python/langchain/middleware/custom

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

### node-style

In [3]:
from dataclasses import dataclass

@dataclass
class Context:
    user_name: str
    age: int = 99

#### 1. 기술적 관점: 에이전트 실행 데이터 비교

| 구분 | **state** (AgentState) | **runtime** (Runtime) |
| :--- | :--- | :--- |
| **성격** | 내부 데이터 (Internal) | 외부 컨텍스트 (External) |
| **주요 데이터** | 대화 기록, 작업 진행 상황, 카운터 등 | 사용자 ID, 환경 변수, 시스템 설정 등 |
| **변경 여부** | 미들웨어 반환값을 통해 업데이트 가능 | 일반적으로 읽기 전용으로 사용 |
| **사용 목적** | "지금까지 무엇을 했는가?"를 파악 | "누가/어떤 환경에서 실행 중인가?"를 파악 |

#### 2. 비유적 관점: 결재 문서 시스템 비교

| 구분 | **state** (문서 데이터) | **runtime** (결재 시스템 환경) |
| :--- | :--- | :--- |
| **저장 위치** | 결재 문서 파일 그 자체에 저장됨 | 회사 전산망 서버 설정에 저장됨 |
| **가변성** | 결재가 진행될수록 도장이 늘어남 | 결재가 진행되어도 회사명은 변하지 않음 |
| **비유적 의미** | "무엇을(What)" 처리하고 있는가? | "어디서/누가(Where/Who)" 처리하는가? |

In [1]:
from langchain.agents.middleware import before_model

@before_model
def log_before_model(state, runtime):
    print("-" * 30)
    print("state:", state)
    print("runtime", runtime)
    print("-" * 30)
    return None

In [4]:
from langchain.agents import create_agent

agent = create_agent(
    model="google_genai:gemini-3.1-flash-lite", 
    tools=[],
    middleware=[log_before_model],
    context_schema=Context
)

In [5]:
agent.invoke(
    {"messages": [{"role": "user", "content": "내 이름이 뭐야?"}]},
    context=Context(user_name="김일남")
)

------------------------------
state: {'messages': [HumanMessage(content='내 이름이 뭐야?', additional_kwargs={}, response_metadata={}, id='48412488-4afe-4673-b0a5-0c972c1a3b07')]}
runtime Runtime(context=Context(user_name='김일남', age=99), store=None, stream_writer=<function Pregel.stream.<locals>.stream_writer at 0x0000021782CE1940>, previous=None)
------------------------------


{'messages': [HumanMessage(content='내 이름이 뭐야?', additional_kwargs={}, response_metadata={}, id='48412488-4afe-4673-b0a5-0c972c1a3b07'),
  AIMessage(content=[{'type': 'text', 'text': '죄송하지만, 저는 사용자의 개인정보를 저장하거나 기억하지 않기 때문에 지금 당신의 이름을 알지 못합니다. \n\n알려주시면 기억하고, 대화하는 동안 불러드릴게요! 성함이 어떻게 되시나요?', 'extras': {'signature': 'EjQKMgEMOdbHU7k5KYVvezxOBTxaSZHa3I+4jgRJMbjZOpH6Y8hCexC6T5uyvi8HRGjKNhlz'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e7aa2-1137-7661-8522-e32bd7818592-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 6, 'output_tokens': 51, 'total_tokens': 57, 'input_token_details': {'cache_read': 0}})]}

### wrap-style

> context에 스키마를 전달하는 것 ≠ LLM이 그 정보를 아는 것

context는 미들웨어/노드 간 데이터를 공유하는 런타임 저장소입니다.<br>
LLM이 해당 정보를 활용하게 하려면,<br>미들웨어에서 request.override(system_prompt=...)처럼 명시적으로 프롬프트에 주입해야 합니다.

In [7]:
# from langchain.agents.middleware import wrap_model_call
# from langchain.messages import HumanMessage, SystemMessage

# @wrap_model_call
# def inject_user_name(request, handler):
#     print(f"request: {request}")
#     print("-" * 10)
#     return handler(request)


In [8]:
from langchain.agents.middleware import wrap_model_call
from langchain.messages import HumanMessage, SystemMessage

@wrap_model_call
def inject_user_name(request, handler):
    print(f"request: {request}")
    print("-" * 10)

    user_name =request.runtime.context.user_name
    
    if user_name:
        sys_prompt = f"사용자의 이름은 {user_name}입니다."
    else:
        sys_prompt = "사용자의 이름은 알려지지 않았습니다."
    request = request.override(system_prompt=sys_prompt) # 명시적으로 프롬프트에 주입
    
    return handler(request)

In [9]:
from langchain.agents.middleware import after_model

@after_model
def log_after_model(state, runtime):
    print(f"after_model_state: {state}")
    print("-" * 10)
    return None

In [10]:
from langchain.agents import create_agent

agent = create_agent(
    model="google_genai:gemini-3.1-flash-lite", 
    tools=[],
    middleware=[inject_user_name, log_after_model],
    context_schema=Context
)

In [11]:
agent.invoke(
    {"messages": [{"role": "user", "content": "내 이름이 뭐야?"}]},
    context=Context(user_name="김일남")
)

request: ModelRequest(model=ChatGoogleGenerativeAI(profile={}, google_api_key=SecretStr('**********'), model='gemini-3.1-flash-lite', temperature=1.0, client=<google.genai.client.Client object at 0x0000021782D6D5E0>, default_metadata=(), model_kwargs={}), messages=[HumanMessage(content='내 이름이 뭐야?', additional_kwargs={}, response_metadata={}, id='0fabebc9-7284-4bdf-b41d-555c3418c1de')], system_message=None, tool_choice=None, tools=[], response_format=None, state={'messages': [HumanMessage(content='내 이름이 뭐야?', additional_kwargs={}, response_metadata={}, id='0fabebc9-7284-4bdf-b41d-555c3418c1de')]}, runtime=Runtime(context=Context(user_name='김일남', age=99), store=None, stream_writer=<function Pregel.stream.<locals>.stream_writer at 0x000002179A19C9A0>, previous=None), model_settings={})
----------
after_model_state: {'messages': [HumanMessage(content='내 이름이 뭐야?', additional_kwargs={}, response_metadata={}, id='0fabebc9-7284-4bdf-b41d-555c3418c1de'), AIMessage(content=[{'type': 'text', 'tex

{'messages': [HumanMessage(content='내 이름이 뭐야?', additional_kwargs={}, response_metadata={}, id='0fabebc9-7284-4bdf-b41d-555c3418c1de'),
  AIMessage(content=[{'type': 'text', 'text': '사용자의 성함은 김일남 님입니다.', 'extras': {'signature': 'EjQKMgEMOdbHeYrgLRy5O5FfEbm1WfgXpj7whSX9uPyjNQEfgQbusL0qnW6qOeCkwozOUoXd'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e7aa9-5408-77e3-bc51-401ca897d680-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 15, 'output_tokens': 12, 'total_tokens': 27, 'input_token_details': {'cache_read': 0}})]}